In [1]:
import os
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pathlib import Path
from IPython.display import Markdown, display, update_display
import pandas as pd

In [12]:
load_dotenv(override=True)

MODEL= "gpt-4o-mini"
#MODEL = "llama3.2"
#KNOWLEDGE_BASE_PATH= Path(__file__).parent / "knowledge_base"

# Initialize openai client
openai = OpenAI()
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(base_url=ollama_url, api_key='ollama')

In [4]:
class CompanyDetails(BaseModel):
    about: str = Field(description="Markdown about the company and its mission")
    careers: str = Field(description="Markdown about the different roles at the company")
    culture: str = Field(description="Markdown about the culture and values of the company")
    overview: str = Field(description="Markdown providing a general overview about the company")

In [5]:
class ContractDetails(BaseModel):
    name: str = Field(description="Name of the contract")
    description: str = Field(description="Description of the contract")
    partner: str = Field(description="Name of the partner the contract is with")

In [6]:
class EmployeeDetails(BaseModel):
    name: str = Field(description="Name of the employee")
    jobTitle: str = Field(description="Name of the employee job title")
    achievement: str = Field(description="A employee achievement")
    skill: str = Field(description="A unique job related skill")

In [7]:
class ProductDetails(BaseModel):
    name: str = Field(description="Name of the product")
    description: str = Field(description="Description of the product")

In [8]:
class KnowledgeBase(BaseModel):
    company: CompanyDetails = Field(description="A dictionary of markdown pertaining to the company")
    contracts: list[ContractDetails] = Field(description="A list of 10 sample contracts with other companies each returned in Mardown")
    employees: list[EmployeeDetails] = Field(description="A list of 10 sample employees and their roles, achievements and skills each returned in Mardown")
    products: list[ProductDetails] = Field(description="A list of 10 sample products and their features and benefits each returned in Mardown")


In [10]:
def generate_knowledge_base(description: str):
    system_prompt = f"""
    You are a expert assistant for generating the knowledge base for a company. Given a
    description of the company you produce a thorough sample knowledge base to be used by a RAG system 
    for the company that can be used to answer questions about the company, its employees products and services.
    The knowledge base should be broken down into four sections: company, contracts, employees and products.

    The company section should contain four sections returned in markdown:
    about: A brief description of the company and its products and services.
    careers: A list of the company's roles with a brief description.
    culture: A description of the company's culture and values.
    overview: A overview about the company.

    The contracts section should contain 2 sample contracts with other companies each property returned in Mardown format, each contract has three properties:
    name: Name of the contract
    description: Description of the contract
    partner: Name of the partner the contract is with
    
    The employees section should contain 2 sample employees and their roles, achievements and skills each property returned in Mardown format, each employee has four properties:
    name: Name of the employee
    jobTitle: Name of the employee job title
    achievement: A employee achievement
    skill: A unique job related skill 

    The products section should contain 10 sample products and their features and benefits each property returned in Mardown format, each product has two properties:
    name: Name of the product
    description: Description of the product
    """

    user_prompt = f"""
    Generate a sample knowledge base for the company described here: {description}.
    Format separate each section and subsection so that they can be broken up into their own markdown file.
    """

    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt }]
    response = openai.chat.completions.parse(
      model=MODEL,
      messages=messages,
      response_format=KnowledgeBase
    )
    result = response.choices[0].message.content

    knowledgeBase_docs = KnowledgeBase.model_validate_json(result)

    return knowledgeBase_docs

In [13]:
description = """I want to generate a Knowledge base for my toy company, we sell a number of different kids toys. With a very
warm and welcoming culture. We have employees in many different fields like technology, marketing, management and may more"
"""
testDocs = generate_knowledge_base(description)

#Markdown(testDocs)


In [14]:
print(testDocs)

company=CompanyDetails(about="# About\nWe are a leading toy company specializing in a diverse range of high-quality children's toys. Our mission is to inspire creativity, imagination, and learning through play. We offer a variety of products, including educational toys, action figures, dolls, and games, all designed with children's development in mind.", careers='# Careers\nAt our company, we value talent and passion. Here are some of the roles we offer:\n- **Toy Designer**: Responsible for creating innovative and engaging toys full of fun and creativity.\n- **Marketing Specialist**: Focuses on promoting our toys through strategic marketing campaigns.\n- **Product Manager**: Oversees the development and launch of new products.\n- **Sales Representative**: Works to build relationships with retailers and increase product visibility in stores.', culture="# Culture\nOur company's culture is warm and welcoming, fostering an environment where creativity thrives. We believe in collaboration a

In [ ]:
def create_knowledge_base_files(description: str):

    #docs = generate_knowledge_base(description)
    docs = testDocs

    company = docs.company
    contracts = docs.contracts
    employees = docs.employees
    products = docs.products

In [ ]:
    
docs = testDocs
company = docs.company
contracts = docs.contracts
employees = docs.employees
products = docs.products

#print(products)
#for key, value in docs.items():
    #print(f"{key}")

#df = pd.DataFrame(company)
#print(df.shape)
  #df.to_markdown(file_name, index=False)

#for name, value in company:
  #extension = languageExtension[language]
 # with open(f"main.{extension}", "w") as f:
    #f.write(code)


print(company.about)



In [ ]:
# Create Company Directory
try:
  folder_path = os.path.join( "company" )
  os.makedirs(folder_path, exist_ok=True)
except FileExistsError:
  print('Directory {} already exists'.format(folder_path))

# About
try:
  file_path = "company/about.md"
  with open(file_path, 'w') as f:
    f.write(company.about)
    print(f"Successfully wrote to {file_path}")
except FileNotFoundError:
    print(f"Error: The directory for '{file_path}' does not exist.")

# Careers
try:
  file_path = "company/careers.md"
  with open(file_path, 'w') as f:
    f.write(company.careers)
    print(f"Successfully wrote to {file_path}")
except FileNotFoundError:
    print(f"Error: The directory for '{file_path}' does not exist.")

# Culture
try:
  file_path = "company/culture.md"
  with open(file_path, 'w') as f:
    f.write(company.culture)
    print(f"Successfully wrote to {file_path}")
except FileNotFoundError:
    print(f"Error: The directory for '{file_path}' does not exist.")

# Overview
try:
  file_path = "company/overview.md"
  with open(file_path, 'w') as f:
    f.write(company.overview)
    print(f"Successfully wrote to {file_path}")
except FileNotFoundError:
    print(f"Error: The directory for '{file_path}' does not exist.")




In [ ]:
# Create Contracts Directory
try:
  folder_path = os.path.join("contracts" )
  os.makedirs(folder_path, exist_ok=True)
except FileExistsError:
  print('Directory {} already exists'.format(folder_path))

for contract in contracts:
    try:
        file_path = f"contracts/{contract.name}.md"
        with open(file_path, 'w') as f:
            f.write(f"Name: {contract.name} \nDescription: {contract.description} \nPartner: {contract.partner}")
            print(f"Successfully wrote to {file_path}")
    except FileNotFoundError:
        print(f"Error: The directory for '{file_path}' does not exist.")


In [ ]:
# Create Employees Directory
try:
  folder_path = os.path.join("employees" )
  os.makedirs(folder_path, exist_ok=True)
except FileExistsError:
  print('Directory {} already exists'.format(folder_path))

for employee in employees:
    try:
        file_path = f"employees/{employee.name}.md"
        with open(file_path, 'w') as f:
            f.write(f"Name: {employee.name} \nJobTitle: {employee.jobTitle} \nSkill: {employee.skill} \nAchievement: {employee.achievement}")
            print(f"Successfully wrote to {file_path}")
    except FileNotFoundError:
        print(f"Error: The directory for '{file_path}' does not exist.")

In [ ]:
# Create Products Directory
try:
  folder_path = os.path.join("products" )
  os.makedirs(folder_path, exist_ok=True)
except FileExistsError:
  print('Directory {} already exists'.format(folder_path))

for product in products:
    try:
        file_path = f"products/{product.name}.md"
        with open(file_path, 'w') as f:
            f.write(f"Name: {product.name} \nDescription: {product.description}")
            print(f"Successfully wrote to {file_path}")
    except FileNotFoundError:
        print(f"Error: The directory for '{file_path}' does not exist.")

In [ ]:
system_prompt = f"""
    You are a expert assistant responsible for creating a jsonL file for RAG testing
    Basd on submitted knowledge base

    Generate 10 test cases in the following format for a JSONL file:
    "question": "", "keywords": [], "category": "" 

    Here is an example:
    "question": "What is the contract number for DriveSmart Insurance's Carllm agreement?", "keywords": ["CR-2025-E-0078", "DriveSmart"], "reference_answer": "The contract number for DriveSmart Insurance's Carllm agreement is CR-2025-E-0078.", "category": "direct_fact"
    """

user_prompt = f"""
    Generate 10 test in JSONL format for the company knowledge base here:
    {testDocs}

    Respond in JSONL only remove any text that is not a JSONL object
    """

messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt }]

response = openai.chat.completions.create(
      model=MODEL,
      messages=messages,
    )

result = response.choices[0].message.content
    
print(result[0])





{"question": "What roles are available in the company?", "keywords": ["Toy Designer", "Marketing Specialist", "Product Manager", "Sales Representative"], "category": "careers"}
{"question": "What is the mission of the company?", "keywords": ["inspire creativity", "imagination", "learning through play"], "category": "about"}
{"question": "When was the company established?", "keywords": ["2005"], "category": "overview"}
{"question": "What is the Interactive Dollhouse?", "keywords": ["Interactive Dollhouse", "tech-savvy", "learn about household roles"], "category": "products"}
{"question": "What are the key values of the company culture?", "keywords": ["inclusivity", "respect", "collaboration", "open communication"], "category": "culture"}
{"question": "Who is the Lead Toy Designer?", "keywords": ["Jane Doe"], "category": "employees"}
{"question": "What kind of toys does the company offer?", "keywords": ["educational toys", "action figures", "dolls", "games"], "category": "about"}
{"quest

In [28]:
#print(result)

try:
    ##file_path = f"tests.jsonl"
    file_path = Path("../evaluation/tests.jsonl")
    with open(file_path, 'w') as f:
        f.write(result)
        print(f"Successfully wrote to {file_path}")
except FileNotFoundError:
    print(f"Error: The directory for '{file_path}' does not exist.")

Successfully wrote to ..\evaluation\tests.jsonl
